In [1]:
# q1_lesson_pages.py

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

print("Actual lesson pages:", len(documents))
#print("Closest answer choice: 72")

Actual lesson pages: 72


In [3]:
# q2_indexing_searching.py

from gitsource import GithubRepositoryDataReader
import minsearch

query = "How does the agentic loop keep calling the model until it stops?"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

results = index.search(query, num_results=5)

print("Top search results:")
for r in results:
    print(r["filename"])

print("\nClosest answer:")
print(results[0]["filename"])

Top search results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md

Closest answer:
01-agentic-rag/lessons/14-agentic-loop.md


In [6]:
# q3_rag.py

from gitsource import GithubRepositoryDataReader
import minsearch
from openai import OpenAI

MODEL = "gpt-5.4-mini"
query = "How does the agentic loop keep calling the model until it stops?"

client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

results = index.search(query, num_results=5)

context = ""
for doc in results:
    context += f"filename: {doc['filename']}\n"
    context += f"content: {doc['content']}\n\n"

prompt = f"""
You are a course teaching assistant.
Answer the QUESTION using only the CONTEXT.

QUESTION:
{query}

CONTEXT:
{context}
""".strip()

response = client.responses.create(
    model=MODEL,
    input=prompt,
)

print("Answer:")
print(response.output_text)

print("\nInput tokens:")
print(response.usage.input_tokens)

print("\nClosest answer choice: 7000")

Answer:
The loop keeps calling the model with a `while True` loop. On each turn it:

1. sends the full `messages` history to `openai_client.responses.create(...)`
2. checks the response for any `function_call` items
3. runs each tool call and appends the tool output to `messages`
4. sets `has_function_calls = True` if any tool was called
5. breaks only when `has_function_calls == False`

So it stops when the model returns a response with no function calls, meaning it has given a final answer.

Input tokens:
7100

Closest answer choice: 7000


In [7]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(

    repo_owner="DataTalksClub",

    repo_name="llm-zoomcamp",

    commit_id="8c1834d",

    allowed_extensions={"md"},

    filename_filter=lambda path: "/lessons/" in path,

)

files = reader.read()

documents = [file.parse() for file in files]

chunks = chunk_documents(

    documents,

    size=2000,

    step=1000,

)

print("Actual number of chunks:", len(chunks))

print("Closest answer choice: 295")

Actual number of chunks: 295
Closest answer choice: 295


In [8]:
# q5_rag_with_chunking.py

from gitsource import GithubRepositoryDataReader, chunk_documents
import minsearch
from openai import OpenAI

MODEL = "gpt-5.4-mini"
query = "How does the agentic loop keep calling the model until it stops?"

client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)

results = index.search(query, num_results=5)

context = ""
for doc in results:
    context += f"filename: {doc['filename']}\n"
    context += f"content: {doc['content']}\n\n"

prompt = f"""
You are a course teaching assistant.
Answer the QUESTION using only the CONTEXT.

QUESTION:
{query}

CONTEXT:
{context}
""".strip()

response = client.responses.create(
    model=MODEL,
    input=prompt,
)

print("Answer:")
print(response.output_text)

print("\nInput tokens with chunking:")
print(response.usage.input_tokens)

print("\nClosest answer choice: 3x fewer")

Answer:
The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any item is a `function_call`.

- If there is a function call, it runs the tool, appends the result to `messages`, and continues.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stopping rule is: **no function calls this turn means we’re done.**

Input tokens with chunking:
2283

Closest answer choice: 3x fewer


In [9]:

# q6_agent.py

import json
from gitsource import GithubRepositoryDataReader, chunk_documents
import minsearch
from openai import OpenAI

MODEL = "gpt-5.4-mini"

client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)

search_count = 0

def search(query):
    global search_count
    search_count += 1
    results = index.search(query, num_results=5)
    return json.dumps(results, ensure_ascii=False)

tools = [
    {
        "type": "function",
        "name": "search",
        "description": "Search the course lesson pages.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query for course lesson content",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    }
]

instructions = """
You are a course teaching assistant.
Use the search tool to answer the question.
Make multiple searches if needed.
""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=question,
    tools=tools,
)

while True:
    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        break

    tool_outputs = []

    for call in function_calls:
        args = json.loads(call.arguments)
        output = search(args["query"])

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": output,
        })

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=tool_outputs,
        tools=tools,
        previous_response_id=response.id,
    )

print("Agent answer:")
print(response.output_text)

print("\nNumber of search calls:")
print(search_count)

print("\nClosest answer choice: 4")

Agent answer:
The **agentic loop** is a repeated cycle where the **LLM is in charge**:

1. You send the user question plus the conversation history.
2. The model may decide to call a tool, like `search`.
3. You run the tool and send the tool result back to the model.
4. The model can call another tool, refine the query, or stop and give a final answer.
5. This repeats until the model finishes.

In the course, it’s described as a `while` loop that:
- calls the LLM,
- executes any tool calls it returns,
- appends the results to memory,
- and keeps going until there are no more tool calls.

### How that differs from plain RAG

**Plain RAG** is a fixed pipeline:

```python
search_results = search(question)
prompt = build_prompt(question, search_results)
answer = llm(prompt)
```

So in plain RAG:
- search happens **once**
- the query is usually the **original user question**
- the system does **not** decide to search again
- the flow is **fixed and deterministic**

### Key difference

- **R